In [40]:
import pandas as pd
from pathlib import Path

In [41]:
data_dir = Path(r"C:\credit_risk_ml\dataset")
csv_files = sorted(data_dir.glob("*.csv"))
xlsx_files = sorted(data_dir.glob("*.xlsx"))

data_raw = {}

for file in csv_files:
    table_name = file.stem

    try:
        data_raw[table_name] = pd.read_csv(file, encoding="utf-8")
        print(f"{table_name}: utf-8")

    except UnicodeDecodeError:
        data_raw[table_name] = pd.read_csv(file, encoding="latin1")
        print(f"{table_name}: latin1")


for file in xlsx_files:
    table_name = file.stem

    try:
        data_raw[table_name] = pd.read_excel(file, encoding="utf-8")
        print(f"{table_name}: utf-8")

    except UnicodeDecodeError:
        data_raw[table_name] = pd.read_excel(file, encoding="latin1")
        print(f"{table_name}: latin1")

application_test: utf-8
application_train: utf-8
bureau: utf-8
bureau_balance: utf-8
credit_card_balance: utf-8
HomeCredit_columns_description: latin1


In [42]:
df_columns_description = data_raw['HomeCredit_columns_description']
df_columns_description[["Table", "Row", "Description"]]

,Table,Row,Description
0,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample
1,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...
2,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving
3,application_{train|test}.csv,CODE_GENDER,Gender of the client
4,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car
...,...,...,...
214,installments_payments.csv,NUM_INSTALMENT_NUMBER,On which installment we observe payment
215,installments_payments.csv,DAYS_INSTALMENT,When the installment of previous credit was su...
216,installments_payments.csv,DAYS_ENTRY_PAYMENT,When was the installments of previous credit p...
217,installments_payments.csv,AMT_INSTALMENT,What was the prescribed installment amount of ...


In [43]:
df_application_train = data_raw['application_train']
df_application_train.dtypes

SK_ID_CURR                      int64
TARGET                          int64
NAME_CONTRACT_TYPE                str
CODE_GENDER                       str
FLAG_OWN_CAR                      str
                               ...   
AMT_REQ_CREDIT_BUREAU_DAY     float64
AMT_REQ_CREDIT_BUREAU_WEEK    float64
AMT_REQ_CREDIT_BUREAU_MON     float64
AMT_REQ_CREDIT_BUREAU_QRT     float64
AMT_REQ_CREDIT_BUREAU_YEAR    float64
Length: 122, dtype: object

In [44]:
layer_1_table = "application_{train|test}.csv"

layer_1_columns = [
    "TARGET",
    "SK_ID_CURR",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "CODE_GENDER",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "OCCUPATION_TYPE",
]

layer_1_descriptions = (
      df_columns_description.loc[(
              df_columns_description["Table"].eq(layer_1_table)
              & df_columns_description["Row"].isin(layer_1_columns)),
          ["Table", "Row", "Description"],].assign(
            Row=lambda df: pd.Categorical(df["Row"],categories=layer_1_columns,ordered=True,)).sort_values("Row"))

# Exportação para melhor leitura
layer_1_descriptions.to_csv('layer_1_descriptions.txt', index=False)

In [45]:
df_application_train_layer_1 = df_application_train.loc[:, layer_1_columns].copy()

In [46]:
df_application_train_layer_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   TARGET               307511 non-null  int64  
 1   SK_ID_CURR           307511 non-null  int64  
 2   EXT_SOURCE_1         134133 non-null  float64
 3   EXT_SOURCE_2         306851 non-null  float64
 4   EXT_SOURCE_3         246546 non-null  float64
 5   AMT_INCOME_TOTAL     307511 non-null  float64
 6   AMT_CREDIT           307511 non-null  float64
 7   AMT_ANNUITY          307499 non-null  float64
 8   AMT_GOODS_PRICE      307233 non-null  float64
 9   DAYS_BIRTH           307511 non-null  int64  
 10  DAYS_EMPLOYED        307511 non-null  int64  
 11  CODE_GENDER          307511 non-null  str    
 12  NAME_INCOME_TYPE     307511 non-null  str    
 13  NAME_EDUCATION_TYPE  307511 non-null  str    
 14  NAME_FAMILY_STATUS   307511 non-null  str    
 15  OCCUPATION_TYPE      211120 

In [47]:
df_application_train_layer_1['SK_ID_CURR'].duplicated().value_counts()

SK_ID_CURR
False    307511
Name: count, dtype: int64

In [48]:
df_application_train_layer_1.isnull().sum().sort_values(ascending=False)

EXT_SOURCE_1           173378
OCCUPATION_TYPE         96391
EXT_SOURCE_3            60965
EXT_SOURCE_2              660
AMT_GOODS_PRICE           278
AMT_ANNUITY                12
TARGET                      0
SK_ID_CURR                  0
AMT_CREDIT                  0
AMT_INCOME_TOTAL            0
DAYS_EMPLOYED               0
DAYS_BIRTH                  0
CODE_GENDER                 0
NAME_INCOME_TYPE            0
NAME_EDUCATION_TYPE         0
NAME_FAMILY_STATUS          0
dtype: int64

In [49]:
#Taxa média de 8,07% de inadimplência, equivalente a 24,8 mil dos 307.511 clientes tem target = 1.
df_application_train_layer_1['TARGET'].describe()

count    307511.000000
mean          0.080729
std           0.272419
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: TARGET, dtype: float64

In [78]:
null_count = df_application_train_layer_1.isna().sum()
null_percentage = df_application_train_layer_1.isna().mean() * 100

missing_summary = pd.DataFrame({
    "null_count": null_count,
    "null_percentage": null_percentage.round(decimals=2)
})


missing_summary.sort_values("null_percentage", ascending=False)


,null_count,null_percentage
EXT_SOURCE_1,173378,56.38
OCCUPATION_TYPE,96391,31.35
EXT_SOURCE_3,60965,19.83
EXT_SOURCE_2,660,0.21
AMT_GOODS_PRICE,278,0.09
AMT_INCOME_TOTAL,0,0.00
TARGET,0,0.00
SK_ID_CURR,0,0.00
AMT_ANNUITY,12,0.00
AMT_CREDIT,0,0.00
